# Notebook A — Zero-shot environment classification (CLIP, no training)

**Task:** multi-label classification of the **environment** of a frame into the 5 classes
`forest, open_field, water, city` (a frame may have several, e.g. water + forest).

**Approach:** **zero-shot CLIP** — for each class we score a *positive* vs *negative* text prompt
and keep the class if `P(present) >= THRESHOLD`. No training, no segmentation.

Outputs per-frame multi-label predictions to `dataset/eval/env_pred_zeroshot.csv` and the mean
**runtime/frame** to `dataset/eval/runtime_zeroshot.json` for the comparison in `seg_evaluation.ipynb`.


## 1. Setup

In [1]:
import sys, json, time
from pathlib import Path

import numpy as np
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, str(Path.cwd()))
import segmentation_common as sc

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
ENV_CLASSES = sc.CATEGORIES["environment"]   # ['forest','open_field','water','city']
print("Device:", DEVICE, "| environment classes:", ENV_CLASSES)

Device: mps | environment classes: ['forest', 'open_field', 'water', 'industry', 'city']


## 2. Configuration

One *positive* and *negative* prompt per environment class drives an independent (multi-label)
decision. `THRESHOLD` is the probability above which a class is considered present.

In [ ]:
# Ensemble prompts (averaged text embeddings) - beat single prompts on the
# validation split; per-class thresholds tuned on validation (dev_documentation).
ENS_POS = {
    "vegetation": ["a forest", "woods with many trees", "trees along the road",
                   "an open field", "a meadow", "open grassland",
                   "grass, plants or greenery", "a green natural area"],
    "water": ["water", "a river", "a lake", "a canal", "the sea",
              "a body of water", "a waterway"],
    "city": ["a city street", "buildings and houses", "an urban area",
             "a street with houses", "industrial buildings", "a built-up area"],
}
ENS_NEG = {
    "vegetation": ["no plants or greenery", "only buildings and pavement",
                   "an urban area with no vegetation", "water only, no plants"],
    "water": ["no water", "dry land", "a scene with no water", "a dry street"],
    "city": ["open countryside", "nature with no buildings",
             "a forest or field, no buildings"],
}
ENV_THRESHOLDS = {"vegetation": 0.035, "water": 0.39, "city": 0.73}  # 3-class, val-tuned

TEST_IMAGES = Path("../dataset/test_images")
PRED_CSV = Path("../dataset/eval/env_pred_zeroshot.csv")
RUNTIME_JSON = Path("../dataset/eval/runtime_zeroshot.json")


def list_test_images(root: Path):
    exts = {".jpg", ".jpeg", ".png"}
    return sorted(p for p in root.rglob("*")
                  if p.suffix.lower() in exts and "overview" not in p.parts)

## 3. CLIP model

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_NAME = "openai/clip-vit-base-patch32"   # ViT-L/14 tested: not better + ~24x slower
clip = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_NAME)


def _text_proto(prompts):
    """Averaged, renormalized text embedding over an ensemble of templates."""
    with torch.no_grad():
        t = clip_proc(text=list(prompts), return_tensors="pt", padding=True).to(DEVICE)
        out = clip.get_text_features(**t)
        f = getattr(out, "pooler_output", out)
        f = f / f.norm(dim=-1, keepdim=True)
        p = f.mean(0)
        return p / p.norm()


def _clip_image(pil):
    with torch.no_grad():
        out = clip.get_image_features(**clip_proc(images=pil, return_tensors="pt").to(DEVICE))
        f = getattr(out, "pooler_output", out)
        return (f / f.norm(dim=-1, keepdim=True)).squeeze(0)


_ENV_POS = {c: _text_proto(ENS_POS[c]) for c in ENV_CLASSES}
_ENV_NEG = {c: _text_proto(ENS_NEG[c]) for c in ENV_CLASSES}
print("Cached ensemble text prototypes for", len(_ENV_POS), "classes")

## 4. Zero-shot classifier

In [ ]:
def score_environment_zeroshot(image_rgb: np.ndarray) -> dict:
    """Per-class CLIP present-probability (continuous). Threshold in evaluation."""
    ifeat = _clip_image(Image.fromarray(image_rgb))
    out = {}
    for c in ENV_CLASSES:
        sims = ifeat @ torch.stack([_ENV_POS[c], _ENV_NEG[c]]).T   # [pos, neg]
        out[c] = float(torch.softmax(sims * 100.0, dim=0)[0])
    return out


def classify_environment_zeroshot(image_rgb: np.ndarray) -> dict:
    """Binary multi-label using the tuned per-class ENV_THRESHOLDS."""
    s = score_environment_zeroshot(image_rgb)
    return {c: int(s[c] >= ENV_THRESHOLDS[c]) for c in ENV_CLASSES}

## 5. Single-frame test

In [ ]:
def _first_test_image():
    imgs = list_test_images(TEST_IMAGES)
    return imgs[0] if imgs else None


sample = _first_test_image()
if sample is None:
    print("No test images under", TEST_IMAGES, "- run scripts/build_test_dataset.py.")
else:
    img = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
    print(sample.name, "->", classify_environment_zeroshot(img))
    plt.imshow(img); plt.axis("off"); plt.show()

## 6. Predict over the test set + runtime

Writes one row per image (`image` + a 0/1 column per class) and the mean runtime/frame.

In [ ]:
def run_zeroshot_testset() -> pd.DataFrame:
    imgs = list_test_images(TEST_IMAGES)
    if not imgs:
        print("No test images found - run scripts/build_test_dataset.py first.")
        return pd.DataFrame()

    rows, t0 = [], time.perf_counter()
    for fp in imgs:
        img = cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB)
        rel = str(fp.relative_to(TEST_IMAGES))
        rows.append({"filename": rel, **score_environment_zeroshot(img)})
    ms = (time.perf_counter() - t0) / len(imgs) * 1000

    df = pd.DataFrame(rows)
    PRED_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(PRED_CSV, index=False)
    json.dump({"method": "zeroshot", "model": "clip-vit-b-32", "value": "softmax_prob",
               "ms_per_frame": ms, "n": len(imgs)},
              open(RUNTIME_JSON, "w"))
    print(f"Saved {len(df)} predictions -> {PRED_CSV}  |  {ms:.1f} ms/frame")
    return df


predictions = run_zeroshot_testset()
predictions.head()